# Exercice 5 : Trier et corréler les alertes

> **En une phrase :** plusieurs outils lèvent chacun leurs alertes pour **la même attaque**. On les regroupe en quelques « méga-alertes » (incidents), puis on les **trie** pour savoir par où commencer.

**Le scénario.** Votre SIEM reçoit les alertes de quatre règles différentes sur le serveur SSH. Votre file d'attente déborde. Votre travail : la ramener à une liste courte et priorisée, comme le ferait le moteur de corrélation d'un SIEM (Wazuh, Splunk, Sentinel…).

**La source.** Le journal SSH de l'exercice 1 (*Loghub*).

**Le plan.**
1. Générer les alertes (quatre « outils »).
2. Mesurer la file d'attente de l'analyste.
3. Corréler : par IP et par proximité dans le temps.
4. Trier : quels incidents en premier ?

## Étape 0 : exécutez cette cellule en premier
Le notebook est **autonome** : il fonctionne tel quel sur Colab (ouvert depuis Drive) ou en local. Elle doit afficher **Prêt**.

In [1]:
import re, requests
import pandas as pd

URL_SSH = "https://raw.githubusercontent.com/logpai/loghub/master/OpenSSH/OpenSSH_2k.log"

def charger_ssh():
    """Télécharge le journal SSH et le range dans un tableau : une ligne = un événement."""
    lignes = requests.get(URL_SSH, timeout=30).text.splitlines()
    motif = re.compile(r"^(\w{3}\s+\d+ \d\d:\d\d:\d\d) \S+ sshd\[\d+\]: (.*)$")
    evenements = []
    for ligne in lignes:
        m = motif.match(ligne)
        if m:
            ip = re.search(r"\d+\.\d+\.\d+\.\d+", m.group(2))
            evenements.append({"heure": pd.to_datetime("2016 " + m.group(1), format="%Y %b %d %H:%M:%S"),  # l'année n'est pas écrite dans le journal
                               "message": m.group(2),
                               "ip": ip.group() if ip else None})
    return pd.DataFrame(evenements), lignes

evenements, _ = charger_ssh()
pd.set_option("display.max_colwidth", 60)
print('Prêt')

Prêt


## Étape 1 : Générer les alertes
**Ce qu'on fait :** quatre règles, comme quatre outils branchés au SIEM. Chacune a une **gravité** (1 = faible, 4 = critique).

| Règle | Déclencheur | Gravité |
|---|---|---|
| R1 Nom DNS incohérent | `POSSIBLE BREAK-IN ATTEMPT` | 1 |
| R2 Compte inexistant | `Invalid user` | 2 |
| R3 Échec sur root | `Failed password for root` | 2 |
| R4 Force brute | 5 échecs ou plus en 2 minutes (ex. 2) | 3 |
| R5 Connexion réussie | `Accepted` | 4 si l'IP a échoué avant, sinon 2 |

In [2]:
alertes = []
def alerte(e, regle, gravite):
    alertes.append({"heure": e["heure"], "ip": e["ip"], "règle": regle, "gravité": gravite})

echecs_par_ip = {}
for _, e in evenements.sort_values("heure").iterrows():
    m = e["message"]
    if "BREAK-IN" in m:                          alerte(e, "R1 nom DNS incohérent", 1)
    if m.startswith("Invalid user"):             alerte(e, "R2 compte inexistant", 2)
    if m.startswith("Failed password for root"): alerte(e, "R3 échec sur root", 2)
    if m.startswith("Failed password"):
        recents = [t for t in echecs_par_ip.get(e["ip"], []) if e["heure"] - t <= pd.Timedelta("2min")] + [e["heure"]]
        echecs_par_ip[e["ip"]] = recents
        if len(recents) >= 5:                    alerte(e, "R4 force brute", 3)
    if m.startswith("Accepted"):
        alerte(e, "R5 connexion réussie", 4 if e["ip"] in echecs_par_ip else 2)

alertes = pd.DataFrame(alertes)
alertes["règle"].value_counts()

règle
R4 force brute           451
R3 échec sur root        368
R2 compte inexistant     113
R1 nom DNS incohérent     85
R5 connexion réussie       1
Name: count, dtype: int64

**👀 Ce qu'on doit voir :** **1 018 alertes**. La règle R4 (force brute) se déclenche **à chaque essai** au-delà du 5e : **451 alertes** pour une poignée d'attaques.

## Étape 2 : La file d'attente de l'analyste
**Ce qu'on fait :** si chaque alerte demande **2 minutes** de triage, combien de temps faut-il ?

In [3]:
duree_journal = evenements["heure"].max() - evenements["heure"].min()
minutes_par_alerte = 2                                              # <-- MODIFIABLE
print(f"{len(alertes)} alertes en {duree_journal}")
print(f"Une alerte toutes les {duree_journal.total_seconds() / len(alertes):.0f} secondes")
print(f"Triage : {len(alertes) * minutes_par_alerte / 60:.0f} heures de travail pour 4 heures de journal, sur UN serveur")

1018 alertes en 0 days 04:08:59
Une alerte toutes les 15 secondes
Triage : 34 heures de travail pour 4 heures de journal, sur UN serveur


**👀 Ce qu'on doit voir :** **une alerte toutes les 15 secondes**, et **≈ 34 heures** de triage pour 4 heures de journal. Impossible à tenir, et ce n'est qu'un serveur.

## Étape 3 : Corréler
**Ce qu'on fait :** deux règles de corrélation du cours :
- **par source** : les alertes de la **même IP** vont ensemble ;
- **temporelle** : si une IP se tait plus de **30 minutes**, ce qui suit est un **nouvel** incident.

In [4]:
silence_max = pd.Timedelta("30min")                                 # <-- MODIFIABLE

alertes = alertes.sort_values(["ip", "heure"])
nouvel_incident = (alertes["ip"] != alertes["ip"].shift()) | (alertes["heure"].diff() > silence_max)
alertes["incident"] = nouvel_incident.cumsum()

incidents = alertes.groupby("incident").agg(
    ip=("ip", "first"), debut=("heure", "min"), fin=("heure", "max"),
    nb_alertes=("règle", "size"), gravite_max=("gravité", "max"),
    regles=("règle", lambda r: ", ".join(sorted({x[:2] for x in r}))))
incidents["durée"] = incidents["fin"] - incidents["debut"]

print(f"{len(alertes)} alertes → {len(incidents)} incidents (réduction de {1 - len(incidents) / len(alertes):.0%})")

1018 alertes → 32 incidents (réduction de 97%)


**👀 Ce qu'on doit voir :** 1 018 alertes deviennent **32 incidents**, soit une réduction de **97 %**. L'analyste lit une ligne par attaque au lieu de centaines.

## Étape 4 : Trier
**Ce qu'on fait :** un score de priorité simple : la **gravité maximale** d'abord, puis le **nombre de règles différentes** déclenchées (plusieurs outils qui voient la même chose = plus de confiance), puis le volume.

In [5]:
incidents["nb_regles"] = incidents["regles"].str.count("R")
file = incidents.sort_values(["gravite_max", "nb_regles", "nb_alertes"], ascending=False)
colonnes = ["ip", "debut", "durée", "nb_alertes", "regles", "gravite_max"]
display(file[colonnes].head(8))
print("Et la connexion réussie, à quel rang ?", list(file.index).index(file[file["regles"].str.contains("R5")].index[0]) + 1, "sur", len(file))
file[file["regles"].str.contains("R5")][colonnes]

,ip,debut,durée,nb_alertes,regles,gravite_max
incident,,,,,,
19,187.141.143.180,2016-12-10 09:12:46,0 days 00:07:16,231,"R1, R2, R3, R4",3
17,183.62.140.253,2016-12-10 10:54:27,0 days 00:10:16,567,"R2, R3, R4",3
4,103.99.0.122,2016-12-10 09:11:20,0 days 00:01:24,53,"R2, R3, R4",3
8,112.95.230.3,2016-12-10 07:27:52,0 days 00:00:59,48,"R2, R3, R4",3
5,103.99.0.122,2016-12-10 11:03:37,0 days 00:01:08,26,"R2, R3, R4",3
24,5.188.10.180,2016-12-10 08:24:32,0 days 00:01:52,23,"R2, R4",3
18,185.190.58.151,2016-12-10 09:07:23,0 days 00:05:36,20,"R2, R4",3
11,123.235.32.19,2016-12-10 07:32:27,0 days 00:01:56,10,"R3, R4",3


Et la connexion réussie, à quel rang ? 19 sur 32


,ip,debut,durée,nb_alertes,regles,gravite_max
incident,,,,,,
9,119.137.62.142,2016-12-10 09:32:20,0 days,1,R5,2


**👀 Ce qu'on doit voir :** en tête, 187.141.143.180 (les **4 règles** R1 à R4), puis 183.62.140.253 (**567 alertes**). Remarquez 103.99.0.122 **deux fois** (09:11 et 11:03) : elle s'est tue plus de 30 minutes, donc deux incidents. La **connexion réussie** de `fztu` (119.137.62.142) n'arrive qu'au **19e rang sur 32** : une seule alerte, gravité 2, car l'IP n'avait jamais échoué avant.

**❓ Question pour la classe :** l'incident 119.137.62.142 ne contient qu'**une** alerte, de gravité 2. 183.62.140.253 en contient des centaines, de gravité 3. Lequel est le plus important ?

*Réponse : 183.62.140.253 fait du bruit mais **a échoué** : aucune connexion réussie. La seule alerte qui décrit un accès **réel** au serveur est celle de 119.137.62.142. Si c'était l'attaquant, ce serait de loin l'incident le plus grave. Le volume d'alertes mesure le **bruit**, pas l'**impact**. Un bon tri ne se contente pas de compter : l'analyste doit vérifier cette connexion (exercice 6).*

**❓ Deuxième question :** ici, on a corrélé **une seule source**. Que gagnerait-on avec la corrélation **multi-source** du cours (courriel + clic + VPN) ?

*Réponse : un attaquant intelligent ne fait pas de force brute bruyante. Il envoie un courriel d'hameçonnage (alerte faible dans la passerelle courriel), l'employé clique (alerte faible dans le proxy), puis une connexion VPN arrive d'un pays inhabituel (alerte faible dans le VPN). **Chaque alerte seule est ignorée ; ensemble, elles racontent une attaque.** Pour les relier, il faut un attribut commun (l'utilisateur, la machine) et une fenêtre de temps : c'est le problème ouvert, en recherche comme en industrie, présenté dans le cours.*

## Ce qu'il faut retenir
- Plusieurs outils → plusieurs alertes pour **la même attaque**. Sans corrélation, l'analyste refait le même travail des centaines de fois.
- Deux corrélations simples (**même source**, **proche dans le temps**) suffisent ici à éliminer ≈ 97 % de la file.
- **Trier ≠ compter** : l'incident le plus bruyant n'est pas forcément le plus grave. Une seule connexion réussie compte plus que 500 échecs.
- Le **triage** décide : faux positif (on ferme), vrai positif simple (on traite), ou trop complexe (on **escalade** au niveau 2).

**Exercice suivant :** investiguer les deux incidents en tête de liste, et rédiger le rapport.